In [ ]:
!pip install shapely matplotlib numpy pandas

In [ ]:
# ==================== 实例分割准确性验证（单页图 + PDF合并 + Panel标签）====================
import json
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import Polygon
from collections import defaultdict
import pandas as pd
from PIL import Image

# -------------------- matplotlib全局参数 --------------------
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.weight'] = 'normal'
plt.rcParams['axes.labelweight'] = 'normal'
plt.rcParams['axes.titleweight'] = 'normal'
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['lines.linewidth'] = 0.5
plt.rcParams['axes.linewidth'] = 0.5
plt.rcParams['xtick.major.width'] = 0.5
plt.rcParams['ytick.major.width'] = 0.5
plt.rcParams['grid.linewidth'] = 0.5
plt.rcParams['hatch.linewidth'] = 0.5

# -------------------- 颜色定义 --------------------
COLOR_RED = "#E89B9B"
COLOR_BLUE = "#7FACCF"
COLOR_GREEN = "#9EC29E"
COLOR_GRAY_LIGHT = "#F0F0F0"

label_colors = {"seed": COLOR_RED, "root": COLOR_GREEN, "leaf": COLOR_BLUE}

# -------------------- 输入文件夹 --------------------
pred_folder = input("预测JSON文件夹路径: ").strip().strip('"').strip("'")
gt_folder = input("人工矫正JSON文件夹路径: ").strip().strip('"').strip("'")

if not os.path.exists(pred_folder) or not os.path.exists(gt_folder):
    raise FileNotFoundError("文件夹路径不存在")

output_dir = os.path.join(os.path.commonpath([pred_folder, gt_folder]), "seg_evaluation_results")
os.makedirs(output_dir, exist_ok=True)
print(f"输出目录: {output_dir}")

# -------------------- 匹配JSON文件 --------------------
def build_name_dict(file_list):
    return {os.path.splitext(os.path.basename(f))[0].lower(): f for f in file_list}

pred_dict = build_name_dict(glob.glob(os.path.join(pred_folder, "*.json")))
gt_dict = build_name_dict(glob.glob(os.path.join(gt_folder, "*.json")))
common_names = sorted(set(pred_dict.keys()) & set(gt_dict.keys()))
if not common_names:
    raise ValueError("未找到匹配的JSON文件，请检查文件名一致性。")
matched_pairs = [(name, pred_dict[name], gt_dict[name]) for name in common_names]
print(f"匹配图像数量: {len(matched_pairs)}")

# -------------------- 多边形评估函数 --------------------
def load_polygons_from_json(json_path):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    polygons = defaultdict(list)
    for shape in data.get("shapes", []):
        label = shape["label"]
        points = shape["points"]
        if len(points) < 3:
            continue
        poly = Polygon(points)
        if poly.is_valid and not poly.is_empty:
            polygons[label].append(poly)
    return polygons

def compute_iou(poly1, poly2):
    if not poly1.intersects(poly2):
        return 0.0
    inter = poly1.intersection(poly2).area
    union = poly1.union(poly2).area
    return inter / union if union > 0 else 0.0

def match_polygons(pred_polys, gt_polys, iou_thresh=0.5):
    if not pred_polys or not gt_polys:
        return len(pred_polys), len(pred_polys), len(gt_polys), []
    iou_matrix = np.zeros((len(pred_polys), len(gt_polys)))
    for i, p in enumerate(pred_polys):
        for j, g in enumerate(gt_polys):
            iou_matrix[i, j] = compute_iou(p, g)
    candidates = [(i, j, iou_matrix[i, j]) for i in range(len(pred_polys))
                  for j in range(len(gt_polys)) if iou_matrix[i, j] >= iou_thresh]
    candidates.sort(key=lambda x: x[2], reverse=True)
    matched_pred, matched_gt = set(), set()
    matched_ious = []
    for i, j, iou in candidates:
        if i not in matched_pred and j not in matched_gt:
            matched_pred.add(i)
            matched_gt.add(j)
            matched_ious.append(iou)
    return len(matched_pred), len(pred_polys)-len(matched_pred), len(gt_polys)-len(matched_gt), matched_ious

# -------------------- 计算每类指标 --------------------
class_metrics = defaultdict(lambda: {"tp":0, "fp":0, "fn":0, "ious":[]})
for _, pred_path, gt_path in matched_pairs:
    pred_polys = load_polygons_from_json(pred_path)
    gt_polys = load_polygons_from_json(gt_path)
    all_labels = set(pred_polys.keys()) | set(gt_polys.keys())
    for label in all_labels:
        tp, fp, fn, ious = match_polygons(pred_polys.get(label, []), gt_polys.get(label, []))
        class_metrics[label]["tp"] += tp
        class_metrics[label]["fp"] += fp
        class_metrics[label]["fn"] += fn
        class_metrics[label]["ious"].extend(ious)

summary = {}
for label, m in class_metrics.items():
    tp, fp, fn = m["tp"], m["fp"], m["fn"]
    prec = tp/(tp+fp) if tp+fp>0 else 0
    rec = tp/(tp+fn) if tp+fn>0 else 0
    f1 = 2*prec*rec/(prec+rec) if prec+rec>0 else 0
    mean_iou = np.mean(m["ious"]) if m["ious"] else 0
    summary[label] = {"precision":prec, "recall":rec, "f1":f1, "mean_iou":mean_iou,
                      "tp":tp, "fp":fp, "fn":fn, "iou_list":m["ious"]}

# -------------------- 四张图依次输出PNG --------------------
labels = list(summary.keys())
x = np.arange(len(labels))
width = 0.25
png_files = []
panel_labels = ['A','B','C','D']

# ---- 图1: Precision / Recall / F1 ----
fig, ax = plt.subplots(figsize=(6,6), dpi=300)
ax.bar(x - width, [summary[l]["precision"] for l in labels], width, color=COLOR_RED, label='Precision', linewidth=0.5, edgecolor='black')
ax.bar(x, [summary[l]["recall"] for l in labels], width, color=COLOR_BLUE, label='Recall', linewidth=0.5, edgecolor='black')
ax.bar(x + width, [summary[l]["f1"] for l in labels], width, color=COLOR_GREEN, label='F1 Score', linewidth=0.5, edgecolor='black')
ax.set_ylabel('Score', fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0,1.05)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(loc='upper left', bbox_to_anchor=(1.02,1), frameon=False)
fig.text(0.02, 0.95, panel_labels[0], fontsize=16, fontweight='bold')  # Panel A
fig.tight_layout()
file1 = os.path.join(output_dir, "fig1_precision_recall_f1.png")
fig.savefig(file1, dpi=300)
png_files.append(file1)
plt.close(fig)

# ---- 图2: Mean IoU ----
fig, ax = plt.subplots(figsize=(6,6), dpi=300)
iou_vals = [summary[l]["mean_iou"] for l in labels]
bars = ax.bar(labels, iou_vals, width=0.6, color=[label_colors[l] for l in labels], alpha=0.8, linewidth=0.5, edgecolor='black')
ax.set_ylabel('IoU', fontsize=11)
ax.set_ylim(0,1.05)
for bar, val in zip(bars, iou_vals):
    ax.text(bar.get_x()+bar.get_width()/2, min(val+0.02,1.02), f'{val:.3f}', ha='center', fontsize=8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(loc='upper left', bbox_to_anchor=(1.02,1), frameon=False)
fig.text(0.02, 0.95, panel_labels[1], fontsize=16, fontweight='bold')  # Panel B
fig.tight_layout()
file2 = os.path.join(output_dir, "fig2_mean_iou.png")
fig.savefig(file2, dpi=300)
png_files.append(file2)
plt.close(fig)

# ---- 图3: IoU箱线图 ----
fig, ax = plt.subplots(figsize=(6,6), dpi=300)
iou_data = [summary[l]["iou_list"] for l in labels if summary[l]["iou_list"]]
box_labels = [l for l in labels if summary[l]["iou_list"]]
if iou_data:
    bp = ax.boxplot(iou_data, tick_labels=box_labels, patch_artist=True, widths=0.6,
                    boxprops=dict(linewidth=0.5, facecolor=COLOR_GRAY_LIGHT, edgecolor='black'),
                    whiskerprops=dict(linewidth=0.5), capprops=dict(linewidth=0.5),
                    medianprops=dict(linewidth=0.5, color='black'),
                    flierprops=dict(marker='o', markersize=2, linewidth=0.5))
    for patch, lbl in zip(bp['boxes'], box_labels):
        patch.set_facecolor(label_colors[lbl])
        patch.set_alpha(0.7)
ax.set_ylabel('IoU', fontsize=11)
ax.set_ylim(0,1.05)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
fig.text(0.02, 0.95, panel_labels[2], fontsize=16, fontweight='bold')  # Panel C
fig.tight_layout()
file3 = os.path.join(output_dir, "fig3_iou_boxplot.png")
fig.savefig(file3, dpi=300)
png_files.append(file3)
plt.close(fig)

# ---- 图4: TP/FP/FN堆叠柱状图 ----
fig, ax = plt.subplots(figsize=(6,6), dpi=300)
tp_vals = [summary[l]["tp"] for l in labels]
fp_vals = [summary[l]["fp"] for l in labels]
fn_vals = [summary[l]["fn"] for l in labels]
ax.bar(x, tp_vals, width=0.6, color=COLOR_BLUE, label='TP', linewidth=0.5, edgecolor='black')
ax.bar(x, fp_vals, width=0.6, bottom=tp_vals, color=COLOR_RED, label='FP', linewidth=0.5, edgecolor='black')
ax.bar(x, fn_vals, width=0.6, bottom=np.array(tp_vals)+np.array(fp_vals), color=COLOR_GREEN, label='FN', linewidth=0.5, edgecolor='black')
ax.set_ylabel('Count', fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(loc='upper left', bbox_to_anchor=(1.02,1), frameon=False)
fig.text(0.02, 0.95, panel_labels[3], fontsize=16, fontweight='bold')  # Panel D
fig.tight_layout()
file4 = os.path.join(output_dir, "fig4_tp_fp_fn.png")
fig.savefig(file4, dpi=300)
png_files.append(file4)
plt.close(fig)

# -------------------- 合并PNG为PDF --------------------
images = [Image.open(png).convert("RGB") for png in png_files]
pdf_path = os.path.join(output_dir, "overall_evaluation.pdf")
images[0].save(pdf_path, save_all=True, append_images=images[1:])
print(f"PDF合并完成: {pdf_path}")

# -------------------- 保存CSV报告 --------------------
report_df = pd.DataFrame([
    {"Class": l, "TP": s["tp"], "FP": s["fp"], "FN": s["fn"],
     "Precision": s["precision"], "Recall": s["recall"], "F1": s["f1"],
     "Mean_IoU": s["mean_iou"], "Num_Matches": len(s["iou_list"])}
    for l, s in summary.items()
])
csv_path = os.path.join(output_dir, "segmentation_evaluation_report.csv")
report_df.to_csv(csv_path, index=False)
print(f"CSV报告已保存: {csv_path}")

In [ ]:
# ==================== YOLOv11-seg vs 传统CV 对比可视化 ====================
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
import os

# -------------------- matplotlib全局参数 --------------------
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.weight"] = "normal"
plt.rcParams["axes.labelweight"] = "normal"
plt.rcParams["axes.titleweight"] = "normal"
plt.rcParams["legend.fontsize"] = 10
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["xtick.labelsize"] = 12
plt.rcParams["ytick.labelsize"] = 12
plt.rcParams["lines.linewidth"] = 0.5
plt.rcParams["axes.linewidth"] = 0.5
plt.rcParams["xtick.major.width"] = 0.5
plt.rcParams["ytick.major.width"] = 0.5
plt.rcParams["grid.linewidth"] = 0.5
plt.rcParams["hatch.linewidth"] = 0.5

# -------------------- 颜色定义 --------------------
COLOR_RED = "#E89B9B"
COLOR_BLUE = "#7FACCF"
COLOR_GREEN = "#9EC29E"
COLOR_GRAY_LIGHT = "#F0F0F0"
COLOR_ORANGE = "#F4A582"
COLOR_PURPLE = "#B8A8D8"

label_colors = {"seed": COLOR_RED, "root": COLOR_GREEN, "leaf": COLOR_BLUE}

# -------------------- 数据定义 --------------------
# YOLOv11-seg 数据
dl_data = {
    "seed": {"TP": 3163, "FP": 99, "FN": 138, "Precision": 0.969650521152667,
             "Recall": 0.9581944865192366, "F1": 0.9638884656407131, "Mean_IoU": 0.6834115185301336},
    "root": {"TP": 1469, "FP": 43, "FN": 103, "Precision": 0.9715608465608465,
             "Recall": 0.9344783715012722, "F1": 0.9526588845654993, "Mean_IoU": 0.7663032669019986},
    "leaf": {"TP": 972, "FP": 35, "FN": 41, "Precision": 0.9652432969215492,
             "Recall": 0.9595261599210266, "F1": 0.9623762376237625, "Mean_IoU": 0.8525254288706555}
}

# 传统CV技术数据
cv_data = {
    "seed": {"TP": 935, "FP": 2188, "FN": 2366, "Precision": 0.2993916106308037,
             "Recall": 0.2832475007573463, "F1": 0.29109589041095896, "Mean_IoU": 0.6821292275617442},
    "root": {"TP": 1184, "FP": 420, "FN": 388, "Precision": 0.7381546134663342,
             "Recall": 0.7531806615776081, "F1": 0.7455919395465994, "Mean_IoU": 0.6157137911402079},
    "leaf": {"TP": 971, "FP": 85, "FN": 45, "Precision": 0.9195075757575758,
             "Recall": 0.9557086614173228, "F1": 0.9372586872586872, "Mean_IoU": 0.8658597223253609}
}

# -------------------- 创建输出目录 --------------------
output_dir = "dl_vs_cv_comparison"
os.makedirs(output_dir, exist_ok=True)
print(f"输出目录: {output_dir}")

labels = ["seed", "root", "leaf"]
x = np.arange(len(labels))
width = 0.35
png_files = []
panel_labels = ['A', 'B', 'C', 'D', 'E', 'F']

# ==================== 图1: Precision对比 ====================
fig, ax = plt.subplots(figsize=(6, 6), dpi=300)
dl_prec = [dl_data[l]["Precision"] for l in labels]
cv_prec = [cv_data[l]["Precision"] for l in labels]

bars1 = ax.bar(x - width/2, dl_prec, width, label='YOLOv11-seg',
               color=COLOR_BLUE, alpha=0.8, linewidth=0.5, edgecolor='black')
bars2 = ax.bar(x + width/2, cv_prec, width, label='Traditional CV',
               color=COLOR_ORANGE, alpha=0.8, linewidth=0.5, edgecolor='black')

ax.set_title('Precision Comparison', fontsize=13, pad=10)
ax.set_ylabel('Precision', fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0, 1.15)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), frameon=False)
ax.axhline(y=0, color='black', linewidth=0.5)

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{height:.3f}', ha='center', va='bottom', fontsize=8)

fig.text(0.02, 0.95, panel_labels[0], fontsize=16, fontweight='bold')
fig.tight_layout()
file1 = os.path.join(output_dir, "fig1_precision_comparison.png")
fig.savefig(file1, dpi=300, bbox_inches='tight')
png_files.append(file1)
plt.close(fig)
print(f"已生成: {file1}")

# ==================== 图2: Recall对比 ====================
fig, ax = plt.subplots(figsize=(6, 6), dpi=300)
dl_rec = [dl_data[l]["Recall"] for l in labels]
cv_rec = [cv_data[l]["Recall"] for l in labels]

bars1 = ax.bar(x - width/2, dl_rec, width, label='YOLOv11-seg',
               color=COLOR_GREEN, alpha=0.8, linewidth=0.5, edgecolor='black')
bars2 = ax.bar(x + width/2, cv_rec, width, label='Traditional CV',
               color=COLOR_PURPLE, alpha=0.8, linewidth=0.5, edgecolor='black')

ax.set_title('Recall Comparison', fontsize=13, pad=10)
ax.set_ylabel('Recall', fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0, 1.15)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), frameon=False)
ax.axhline(y=0, color='black', linewidth=0.5)

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{height:.3f}', ha='center', va='bottom', fontsize=8)

fig.text(0.02, 0.95, panel_labels[1], fontsize=16, fontweight='bold')
fig.tight_layout()
file2 = os.path.join(output_dir, "fig2_recall_comparison.png")
fig.savefig(file2, dpi=300, bbox_inches='tight')
png_files.append(file2)
plt.close(fig)
print(f"已生成: {file2}")

# ==================== 图3: F1 Score对比 ====================
fig, ax = plt.subplots(figsize=(6, 6), dpi=300)
dl_f1 = [dl_data[l]["F1"] for l in labels]
cv_f1 = [cv_data[l]["F1"] for l in labels]

bars1 = ax.bar(x - width/2, dl_f1, width, label='YOLOv11-seg',
               color=COLOR_RED, alpha=0.8, linewidth=0.5, edgecolor='black')
bars2 = ax.bar(x + width/2, cv_f1, width, label='Traditional CV',
               color=COLOR_GRAY_LIGHT, alpha=0.8, linewidth=0.5, edgecolor='black')

ax.set_title('F1 Score Comparison', fontsize=13, pad=10)
ax.set_ylabel('F1 Score', fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0, 1.15)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), frameon=False)
ax.axhline(y=0, color='black', linewidth=0.5)

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{height:.3f}', ha='center', va='bottom', fontsize=8)

fig.text(0.02, 0.95, panel_labels[2], fontsize=16, fontweight='bold')
fig.tight_layout()
file3 = os.path.join(output_dir, "fig3_f1_comparison.png")
fig.savefig(file3, dpi=300, bbox_inches='tight')
png_files.append(file3)
plt.close(fig)
print(f"已生成: {file3}")

# ==================== 图4: Mean IoU对比 ====================
fig, ax = plt.subplots(figsize=(6, 6), dpi=300)
dl_iou = [dl_data[l]["Mean_IoU"] for l in labels]
cv_iou = [cv_data[l]["Mean_IoU"] for l in labels]

bars1 = ax.bar(x - width/2, dl_iou, width, label='YOLOv11-seg',
               color=COLOR_BLUE, alpha=0.8, linewidth=0.5, edgecolor='black')
bars2 = ax.bar(x + width/2, cv_iou, width, label='Traditional CV',
               color=COLOR_ORANGE, alpha=0.8, linewidth=0.5, edgecolor='black')

ax.set_title('Mean IoU Comparison', fontsize=13, pad=10)
ax.set_ylabel('Mean IoU', fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0, 1.05)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), frameon=False)
ax.axhline(y=0, color='black', linewidth=0.5)

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{height:.3f}', ha='center', va='bottom', fontsize=8)

fig.text(0.02, 0.95, panel_labels[3], fontsize=16, fontweight='bold')
fig.tight_layout()
file4 = os.path.join(output_dir, "fig4_iou_comparison.png")
fig.savefig(file4, dpi=300, bbox_inches='tight')
png_files.append(file4)
plt.close(fig)
print(f"已生成: {file4}")

# ==================== 图5: 综合指标雷达图 ====================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6), dpi=300,
                               subplot_kw=dict(projection='polar'))

categories = ['Precision', 'Recall', 'F1 Score', 'Mean IoU']
N = len(categories)

dl_avg = [
    np.mean([dl_data[l]["Precision"] for l in labels]),
    np.mean([dl_data[l]["Recall"] for l in labels]),
    np.mean([dl_data[l]["F1"] for l in labels]),
    np.mean([dl_data[l]["Mean_IoU"] for l in labels])
]

cv_avg = [
    np.mean([cv_data[l]["Precision"] for l in labels]),
    np.mean([cv_data[l]["Recall"] for l in labels]),
    np.mean([cv_data[l]["F1"] for l in labels]),
    np.mean([cv_data[l]["Mean_IoU"] for l in labels])
]

angles = [n / float(N) * 2 * np.pi for n in range(N)]
dl_avg_plot = dl_avg + dl_avg[:1]
cv_avg_plot = cv_avg + cv_avg[:1]
angles_plot = angles + angles[:1]

# YOLOv11-seg 雷达图
ax1.plot(angles_plot, dl_avg_plot, 'o-', linewidth=2, color=COLOR_BLUE)
ax1.fill(angles_plot, dl_avg_plot, alpha=0.25, color=COLOR_BLUE)
ax1.set_xticks(angles)
ax1.set_xticklabels(categories, fontsize=10)
ax1.set_ylim(0, 1)
ax1.set_title('YOLOv11-seg', fontsize=12, pad=20)
ax1.grid(True, linewidth=0.5)
ax1.spines['polar'].set_linewidth(0.5)

# 传统CV 雷达图
ax2.plot(angles_plot, cv_avg_plot, 'o-', linewidth=2, color=COLOR_ORANGE)
ax2.fill(angles_plot, cv_avg_plot, alpha=0.25, color=COLOR_ORANGE)
ax2.set_xticks(angles)
ax2.set_xticklabels(categories, fontsize=10)
ax2.set_ylim(0, 1)
ax2.set_title('Traditional CV', fontsize=12, pad=20)
ax2.grid(True, linewidth=0.5)
ax2.spines['polar'].set_linewidth(0.5)

fig.text(0.02, 0.95, panel_labels[4], fontsize=16, fontweight='bold', transform=fig.transFigure)
fig.suptitle('Overall Metric Comparison (Average across Classes)', fontsize=12, y=1.01)
fig.tight_layout()
file5 = os.path.join(output_dir, "fig5_radar_comparison.png")
fig.savefig(file5, dpi=300, bbox_inches='tight')
png_files.append(file5)
plt.close(fig)
print(f"已生成: {file5}")

# ==================== 图6: TP/FP/FN对比 ====================
fig, axes = plt.subplots(1, 2, figsize=(12, 6), dpi=300)

# YOLOv11-seg
ax = axes[0]
dl_tp = [dl_data[l]["TP"] for l in labels]
dl_fp = [dl_data[l]["FP"] for l in labels]
dl_fn = [dl_data[l]["FN"] for l in labels]

ax.bar(x, dl_tp, width=0.6, color=COLOR_BLUE, label='TP', linewidth=0.5, edgecolor='black')
ax.bar(x, dl_fp, width=0.6, bottom=dl_tp,
       color=COLOR_RED, label='FP', linewidth=0.5, edgecolor='black')
ax.bar(x, dl_fn, width=0.6, bottom=np.array(dl_tp) + np.array(dl_fp),
       color=COLOR_GREEN, label='FN', linewidth=0.5, edgecolor='black')

ax.set_title('YOLOv11-seg', fontsize=13, pad=10)
ax.set_ylabel('Count', fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(loc='upper right', frameon=False)

# 传统CV
ax = axes[1]
cv_tp = [cv_data[l]["TP"] for l in labels]
cv_fp = [cv_data[l]["FP"] for l in labels]
cv_fn = [cv_data[l]["FN"] for l in labels]

ax.bar(x, cv_tp, width=0.6, color=COLOR_BLUE, label='TP', linewidth=0.5, edgecolor='black')
ax.bar(x, cv_fp, width=0.6, bottom=cv_tp,
       color=COLOR_RED, label='FP', linewidth=0.5, edgecolor='black')
ax.bar(x, cv_fn, width=0.6, bottom=np.array(cv_tp) + np.array(cv_fp),
       color=COLOR_GREEN, label='FN', linewidth=0.5, edgecolor='black')

ax.set_title('Traditional CV', fontsize=13, pad=10)
ax.set_ylabel('Count', fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(loc='upper right', frameon=False)

fig.text(0.02, 0.95, panel_labels[5], fontsize=16, fontweight='bold', transform=fig.transFigure)
fig.suptitle('TP / FP / FN Comparison', fontsize=12, y=1.01)
fig.tight_layout()
file6 = os.path.join(output_dir, "fig6_tp_fp_fn_comparison.png")
fig.savefig(file6, dpi=300, bbox_inches='tight')
png_files.append(file6)
plt.close(fig)
print(f"已生成: {file6}")

# ==================== 合并PNG为PDF ====================
images = [Image.open(png).convert("RGB") for png in png_files]
pdf_path = os.path.join(output_dir, "yolov11seg_vs_cv_comparison.pdf")
images[0].save(pdf_path, save_all=True, append_images=images[1:])
print(f"PDF合并完成: {pdf_path}")

# ==================== 生成对比报告CSV ====================
comparison_data = []
for label in labels:
    for method, data in [("YOLOv11-seg", dl_data), ("Traditional CV", cv_data)]:
        comparison_data.append({
            "Class": label,
            "Method": method,
            "TP": data[label]["TP"],
            "FP": data[label]["FP"],
            "FN": data[label]["FN"],
            "Precision": data[label]["Precision"],
            "Recall": data[label]["Recall"],
            "F1": data[label]["F1"],
            "Mean_IoU": data[label]["Mean_IoU"]
        })

comparison_df = pd.DataFrame(comparison_data)
csv_path = os.path.join(output_dir, "yolov11seg_vs_cv_report.csv")
comparison_df.to_csv(csv_path, index=False)
print(f"CSV报告已保存: {csv_path}")

# ==================== 改进率统计 ====================
improvement_data = []
for label in labels:
    improvement_data.append({
        "Class": label,
        "Precision_Improvement_%": (dl_data[label]["Precision"] - cv_data[label]["Precision"]) / cv_data[label]["Precision"] * 100,
        "Recall_Improvement_%": (dl_data[label]["Recall"] - cv_data[label]["Recall"]) / cv_data[label]["Recall"] * 100,
        "F1_Improvement_%": (dl_data[label]["F1"] - cv_data[label]["F1"]) / cv_data[label]["F1"] * 100,
        "IoU_Improvement_%": (dl_data[label]["Mean_IoU"] - cv_data[label]["Mean_IoU"]) / cv_data[label]["Mean_IoU"] * 100
    })

improvement_df = pd.DataFrame(improvement_data)
improvement_csv = os.path.join(output_dir, "improvement_analysis.csv")
improvement_df.to_csv(improvement_csv, index=False)
print(f"改进率分析已保存: {improvement_csv}")

print(" " + "=" * 60)
print("所有图表和报告已生成到文件夹，未在Notebook中显示图像。")
print("=" * 60)